In [13]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import (
    StateGraph,
    START,
    END
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from finagent.rag.pipeline import ask_rag
from finagent.market.pipeline import analyze_market
from finagent.forecast.pipeline import forecast_stock

In [14]:
# ============================================================
# 1. STATE
# ============================================================

class FinAgentState(TypedDict):

    question: str

    selected_agents: list[str]

    rag_answer: str
    market_answer: str
    forecast_answer: str

    final_answer: str


# ============================================================
# 2. ROUTER OUTPUT
# ============================================================

class RouteDecision(BaseModel):

    agents: list[
        Literal[
            "rag",
            "market",
            "forecast"
        ]
    ] = Field(
        min_length=1
    )

In [15]:
# ============================================================
# 3. ROUTER LLM
# ============================================================

router_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


structured_router = (
    router_llm.with_structured_output(
        RouteDecision
    )
)

In [16]:
# ============================================================
# 4. ROUTER PROMPT
# ============================================================

router_prompt = """
You are the router of FinAgent-X,
a multi-agent financial intelligence system
focused on Malaysian listed companies.

Select ONE OR MORE agents required
to answer the user's question.

Available agents:

rag:
Use for financial news, company events,
announcements, earnings reported in news,
management changes, acquisitions, contracts,
and explanations based on The Edge Malaysia.

market:
Use for current or historical market data
and technical indicators such as price,
returns, volume, RSI, MACD, moving averages,
volatility, momentum, ATR or Bollinger Bands.

forecast:
Use when the user asks about future price
direction, expected return, prediction,
forecast or outlook based on the forecasting model.


Examples:

"What did The Edge report about Maybank?"
→ ["rag"]

"What is Maybank's RSI?"
→ ["market"]

"Predict Maybank's next-day return."
→ ["forecast"]

"Explain Maybank's recent news and technical situation."
→ ["rag", "market"]

"Analyze Maybank using news, technical indicators
and the forecast."
→ ["rag", "market", "forecast"]


Select only the agents genuinely needed.

QUESTION:

{question}
"""

In [17]:
# ============================================================
# 5. ROUTER NODE
# ============================================================

def router_node(
    state: FinAgentState
):

    prompt = router_prompt.format(
        question=state["question"]
    )

    decision = structured_router.invoke(
        prompt
    )

    # Remove possible duplicates
    agents = list(
        dict.fromkeys(
            decision.agents
        )
    )

    return {
        "selected_agents":
            agents
    }

In [18]:
# ============================================================
# 6. RAG AGENT
# ============================================================

def rag_node(
    state: FinAgentState
):

    result = ask_rag(
        state["question"]
    )

    return {
        "rag_answer":
            result["answer"]
    }


# ============================================================
# 7. MARKET AGENT
# ============================================================

def market_node(
    state: FinAgentState
):

    result = analyze_market(
        state["question"]
    )

    return {
        "market_answer":
            result["answer"]
    }


# ============================================================
# 8. FORECAST AGENT
# ============================================================

def forecast_node(
    state: FinAgentState
):

    result = forecast_stock(
        state["question"]
    )

    return {
        "forecast_answer":
            result["answer"]
    }

In [19]:
# ============================================================
# 9. CONDITIONAL ROUTING
# ============================================================

def choose_agents(
    state: FinAgentState
):

    mapping = {
        "rag":
            "rag_agent",

        "market":
            "market_agent",

        "forecast":
            "forecast_agent"
    }

    return [
        mapping[agent]
        for agent
        in state["selected_agents"]
    ]


# ============================================================
# 10. SYNTHESIS LLM
# ============================================================

synthesis_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are the synthesis agent of FinAgent-X.

Your task is to combine the analyses produced
by specialized financial agents into one clear,
coherent answer.

Possible sources:

- News/RAG Agent:
  explains financial news and company events.

- Market Agent:
  interprets market data and technical indicators.

- Forecast Agent:
  provides an experimental model-based estimate
  of the next-session return.

Rules:

1. Use ONLY the agent outputs provided.

2. Do not invent financial facts.

3. Do not invent missing agent results.

4. Clearly distinguish:
   - news/events,
   - technical market signals,
   - forecasting estimates.

5. Forecasts are uncertain quantitative signals,
   not guaranteed future outcomes.

6. Answer the user's original question directly.

7. Be concise but sufficiently explanatory.
"""
        ),

        (
            "human",
            """
USER QUESTION:

{question}


SELECTED AGENTS:

{selected_agents}


NEWS / RAG ANALYSIS:

{rag_answer}


MARKET ANALYSIS:

{market_answer}


FORECAST ANALYSIS:

{forecast_answer}


FINAL SYNTHESIS:
"""
        )
    ]
)


synthesis_chain = (
    synthesis_prompt
    | synthesis_llm
    | StrOutputParser()
)

In [20]:
# ============================================================
# 11. SYNTHESIS NODE
# ============================================================

def synthesis_node(
    state: FinAgentState
):

    final_answer = synthesis_chain.invoke(
        {
            "question":
                state["question"],

            "selected_agents":
                ", ".join(
                    state.get(
                        "selected_agents",
                        []
                    )
                ),

            "rag_answer":
                state.get(
                    "rag_answer",
                    ""
                ),

            "market_answer":
                state.get(
                    "market_answer",
                    ""
                ),

            "forecast_answer":
                state.get(
                    "forecast_answer",
                    ""
                )
        }
    )

    return {
        "final_answer":
            final_answer
    }


In [21]:
# ============================================================
# 12. BUILD GRAPH
# ============================================================

builder = StateGraph(
    FinAgentState
)


builder.add_node(
    "router",
    router_node
)

builder.add_node(
    "rag_agent",
    rag_node
)

builder.add_node(
    "market_agent",
    market_node
)

builder.add_node(
    "forecast_agent",
    forecast_node
)

builder.add_node(
    "synthesis",
    synthesis_node
)


# START → ROUTER

builder.add_edge(
    START,
    "router"
)


# ROUTER → ONE OR MULTIPLE AGENTS

builder.add_conditional_edges(
    "router",
    choose_agents
)


# ALL SPECIALIZED AGENTS → SYNTHESIS

builder.add_edge(
    "rag_agent",
    "synthesis"
)

builder.add_edge(
    "market_agent",
    "synthesis"
)

builder.add_edge(
    "forecast_agent",
    "synthesis"
)


# SYNTHESIS → END

builder.add_edge(
    "synthesis",
    END
)


# ============================================================
# 13. COMPILE
# ============================================================

graph = builder.compile()

In [22]:
print(
    graph.get_graph().draw_ascii()
)

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | router |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [23]:
initial_state = {

    "question":
        "What is Maybank's RSI?",

    "selected_agents": [],

    "rag_answer": "",
    "market_answer": "",
    "forecast_answer": "",

    "final_answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "SELECTED AGENTS:",
    result["selected_agents"]
)


print(
    "\nFINAL ANSWER:\n"
)

print(
    result["final_answer"]
)

SELECTED AGENTS: ['market']

FINAL ANSWER:

As of 2026-11-06, Maybank's 14-day Relative Strength Index (RSI) is 55.61.


In [24]:
initial_state = {

    "question": (
        "Explain Maybank's recent news "
        "and technical situation."
    ),

    "selected_agents": [],

    "rag_answer": "",
    "market_answer": "",
    "forecast_answer": "",

    "final_answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "SELECTED AGENTS:",
    result["selected_agents"]
)


print(
    "\nRAG ANSWER:\n",
    result["rag_answer"]
)


print(
    "\nMARKET ANSWER:\n",
    result["market_answer"]
)


print(
    "\nFINAL ANSWER:\n",
    result["final_answer"]
)

SELECTED AGENTS: ['rag', 'market']

RAG ANSWER:
 Maybank's recent news primarily revolves around its financial performance and outlook:

For 2QFY2025, Malayan Banking Bhd (Maybank) reported a nearly 4% year-on-year rise in net profit to RM2.63 billion, as higher income compensated for increased provisions for bad debts. Net interest income grew 1.1% year-on-year, while non-interest income climbed 18%. The bank declared a first interim dividend of 30 sen per share and reiterated its target for a return on equity (ROE) of at least 11.3% for FY2025. However, Maybank lowered its loan growth guidance for the year to approximately 3% from the previous 5%-6%, citing client caution due to tariff uncertainties leading to delays in planned investments and capital-raising activities [Source 3].

For 3QFY2024, Maybank's net profit increased by 7.6% to RM2.54 billion compared to the previous year, primarily driven by growth in non-interest income which offset a 1.9% decline in net interest income. 

In [25]:
initial_state = {

    "question": (
        "Analyze Maybank using recent news, "
        "technical indicators and the forecast."
    ),

    "selected_agents": [],

    "rag_answer": "",
    "market_answer": "",
    "forecast_answer": "",

    "final_answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "SELECTED AGENTS:",
    result["selected_agents"]
)


print(
    "\nFINAL ANSWER:\n"
)

print(
    result["final_answer"]
)

SELECTED AGENTS: ['rag', 'market', 'forecast']

FINAL ANSWER:

Here's an analysis of Maybank based on recent news, technical indicators, and the forecast:

**Recent News and Company Outlook:**

Maybank has shown mixed financial performance in recent quarters, often driven by non-interest income growth offsetting declines in net interest income.
*   **Q3FY2024:** Net profit rose 7.6% to RM2.54 billion, supported by a 7.5% increase in non-interest income and a 12% rise in Islamic banking income, despite a 1.9% slip in net interest income. Return on Equity (ROE) was 11.1%.
*   **Q2FY2025:** Net profit increased nearly 4% year-on-year to RM2.63 billion, backed by an 18% climb in non-interest income and a 1.1% increase in net interest income. A first interim dividend of 30 sen per share was declared.
*   **Q4FY2023 & Full FY2023:** Net profit grew 8.3% to RM2.39 billion in Q4, mainly from strong gains in insurance and non-interest income. For the full FY2023, net profit climbed 17% to RM9.3

In [26]:
from finagent.graph import graph

In [27]:
initial_state = {

    "question": (
        "Analyze Maybank using recent news, "
        "technical indicators and the forecast."
    ),

    "selected_agents": [],

    "rag_answer": "",
    "market_answer": "",
    "forecast_answer": "",

    "final_answer": ""
}


result = graph.invoke(
    initial_state
)

In [28]:
print(
    "SELECTED AGENTS:",
    result["selected_agents"]
)


print(
    "\nFINAL ANSWER:\n"
)


print(
    result["final_answer"]
)

SELECTED AGENTS: ['rag', 'market', 'forecast']

FINAL ANSWER:

Here's an analysis of Maybank based on recent news, technical indicators, and the latest forecast:

**Recent News and Company Events (from RAG Agent):**

Maybank has shown mixed but generally positive financial performance across recent quarters.
*   **3QFY2024 (ended Sept 30, 2024):** Net profit increased by 7.6% to RM2.54 billion, driven by non-interest income growth which offset a 1.9% decline in net interest income. Islamic banking income rose 12%, and non-interest income was up 7.5%. No dividend was declared for this quarter.
*   **2QFY2025 (ended June 30, 2025):** Net profit rose nearly 4% year-on-year to RM2.63 billion, with higher income despite increased provisions for bad debts. Net interest income grew 1.1%, and non-interest income climbed 18%. A first interim dividend of 30 sen per share was declared.
*   **4QFY2023 (ended Dec 31, 2023):** Net profit grew 8.3% year-on-year to RM2.39 billion, supported by strong 